## CS-SHRED on NOAA OISST Sea-Surface-Temperature data (real buoy sensors)

This notebook runs **CS-SHRED** (Compressed-Sensing Shallow Recurrent Decoder) to reconstruct
the full daily SST field from **20 real buoy time series**.

> da Silva, R.B., Passos, D., Oishi, C.M., Kutz, J.N.
> *CS-SHRED: Enhancing SHRED for Robust Recovery of Spatiotemporal Dynamics*,
> arXiv:2507.22303 (2025). Code: <https://github.com/romulobrito/cs-shred>

**How the pipeline works (and what was fixed w.r.t. the previous version)**

- **Input** `X`: sliding windows of `lags` days of the 20 buoy series, *corrupted* with missing
  samples (missing = exact `0` after scaling, which is what `CSSHREDLAGS` treats as "unobserved").
- **Target** `Y`: the **clean** full SST field (ocean pixels only). The previous version used the
  field after `subsample()` as target, so the model was learning to reproduce zeros.
- Windows are built **inside each year**, and train/valid/test are split **by year**, so no window
  crosses two unrelated years and no overlapping windows leak across splits.
- `MinMaxScaler`s are fitted on the training years only and **actually applied** to inputs and targets.
- Sensor values are scaled to `[0.1, 1]` so that a genuine measurement is never exactly `0`.
- The CS step in `CSSHREDLAGS` only runs when the fraction of zeros in a lag window exceeds
  `n_sparsity_threshold`. With 30 % missing samples the repo default (0.75) would **never** trigger it
  (the model would silently be a plain SHRED), so the threshold is lowered below the missing fraction.

## 0. Setup — clone CS-SHRED and check dependencies

In [14]:
import subprocess, sys
from pathlib import Path

CS_SHRED_DIR = Path("cs-shred")

if not CS_SHRED_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/romulobrito/cs-shred.git", str(CS_SHRED_DIR)],
        check=True,
    )
else:
    print(f"'{CS_SHRED_DIR}' already exists, skipping clone.")

# The venv has no pip: use uv if the CS stack (pylops + spgl1) is missing.
try:
    import pylops, spgl1  # noqa: F401
except ImportError:
    subprocess.run(
        ["uv", "pip", "install", "--python", sys.executable,
         "-r", str(CS_SHRED_DIR / "requirements.txt")],
        check=True,
    )

'cs-shred' already exists, skipping clone.


In [15]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.preprocessing import MinMaxScaler
from skimage.metrics import structural_similarity as ssim
from numpy.lib.stride_tricks import sliding_window_view

# Make the cloned repo importable so we can reuse its model / dataset code.
sys.path.insert(0, str(CS_SHRED_DIR.resolve()))

import models                              # SHRED, CSSHREDLAGS, fit_csshred_model, ...
from processdata import TimeSeriesDataset  # (X, Y) Dataset wrapper exposing .X and .Y

# CSSHREDLAGS runs SPGL1 on the CPU and moves every batch to CPU anyway,
# and the datasets live in CPU memory, so everything stays on CPU.
device = torch.device("cpu")
print(f"Using device: {device}")

SEED = 915
np.random.seed(SEED)
torch.manual_seed(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cpu


## 1. Load the SST fields and the buoy series

`sst_all_ROM.npy` has shape `(years, days, 180*360)` and `buoy_all_ROM.npy` has shape
`(years, days, 20)`. Land pixels are exact zeros in every snapshot, so they are dropped.
Years are shuffled with a fixed seed (as before) **but the time axis inside each year is kept intact**.
Everything is cast to float32 to keep memory manageable (the float64 field alone was ~4.9 GB).

In [16]:
DATA_DIR = Path.cwd() / "../Notebooks/SST/datasets"

X_mm = np.load(DATA_DIR / "sst_all_ROM.npy", mmap_mode="r")        # (years, days, 64800)
S_raw = np.load(DATA_DIR / "buoy_all_ROM.npy").astype(np.float32)  # (years, days, 20)
print(f"SST data   : {X_mm.shape}")
print(f"Buoy data  : {S_raw.shape}\n")

loc = pd.read_csv(DATA_DIR / "stations_location.csv", dtype={"station": str})
for name, lat, lon in zip(loc["station"], loc["lat"], loc["lon"]):
    print(f"Buoy {name} at lat: {lat}, lon: {lon}")

n_years, n_days, n_grid = X_mm.shape
num_sensors = S_raw.shape[2]
assert n_grid == 180 * 360

# Shuffle YEARS only (time order inside each year is preserved).
np.random.seed(42)
year_perm = np.random.permutation(n_years)

X_years = np.empty(X_mm.shape, dtype=np.float32)
for k, y in enumerate(year_perm):
    X_years[k] = X_mm[y]
S_years = S_raw[year_perm]
del S_raw

# Ocean-only state vector (land = exact zeros in every snapshot).
mean_X = X_years.reshape(-1, n_grid).mean(axis=0, dtype=np.float64)
sst_locs = np.where(mean_X != 0)[0]
X_clean = np.ascontiguousarray(X_years[:, :, sst_locs])             # (years, days, n_ocean)
del X_years

n_ocean = X_clean.shape[2]
print(f"\nX_clean: {X_clean.shape}  ({X_clean.nbytes/1e9:.2f} GB)")
print(f"Sensors: {S_years.shape}, NaN in sensors: {int(np.isnan(S_years).sum())}")

SST data   : (26, 365, 64800)
Buoy data  : (26, 365, 20)

Buoy 46035 at lat: 57.034, lon: -177.468
Buoy 46001 at lat: 56.296, lon: -148.027
Buoy 46066 at lat: 52.776, lon: -154.992
Buoy 46006 at lat: 40.73, lon: -137.42
Buoy 46059 at lat: 38.067, lon: -129.895
Buoy 46025 at lat: 33.765, lon: -119.077
Buoy 51001 at lat: 24.475, lon: -162.03
Buoy 51004 at lat: 17.504, lon: -152.197
Buoy 52200 at lat: 13.354, lon: 144.788
Buoy 44013 at lat: 42.346, lon: -70.651
Buoy 41001 at lat: 34.738, lon: -72.503
Buoy 41010 at lat: 28.86, lon: -78.478
Buoy 41046 at lat: 23.84, lon: -68.34
Buoy 41043 at lat: 21.09, lon: -64.864
Buoy 41044 at lat: 21.582, lon: -58.63
Buoy 42057 at lat: 16.975, lon: -81.578
Buoy 42058 at lat: 14.114, lon: -75.949
Buoy 42040 at lat: 29.207, lon: -88.237
Buoy 42001 at lat: 25.922, lon: -89.638
Buoy 42002 at lat: 25.95, lon: -93.78

X_clean: (26, 365, 41027)  (1.56 GB)
Sensors: (26, 365, 20), NaN in sensors: 51379


## 2. Year-based split, scaling, and sensor corruption

- Split by year: 70 % / 15 % / 15 % (random but seeded, thanks to the permutation above).
- `sc_x` (sensors, per sensor, range `[0.1, 1]`) and `sc_y` (field, per pixel) are fitted on **train years only**.
- Corruption follows the CS-SHRED idea of zero-filled missing data, applied to the **sensor inputs**:
  each sensor sample is independently dropped with probability `SENSOR_TIME_DROP` (30 %), and
  optionally whole sensors can be dropped (`SENSOR_DROP_FRAC`, default 0: with only 20 real buoys,
  removing 90 % of them as the paper does for pixels is not meaningful). Genuine `NaN`s in the
  buoy files are treated as additional missing samples.
- The **target stays clean**.

In [17]:
LAGS = 20
SENSOR_TIME_DROP = 0.30     # fraction of sensor samples (day, sensor) set to missing
SENSOR_DROP_FRAC = 0.0      # fraction of whole sensors removed (0 = keep all 20 buoys)

n_train = int(0.70 * n_years)
n_valid = int(0.15 * n_years)
train_years = np.arange(0, n_train)
valid_years = np.arange(n_train, n_train + n_valid)
test_years  = np.arange(n_train + n_valid, n_years)
print(f"years -> train {len(train_years)}, valid {len(valid_years)}, test {len(test_years)}")

# --- scalers (train years only) ---------------------------------------------------------
# Sensors: [0.1, 1] so that a real measurement is never exactly 0 (0 means "missing").
sc_x = MinMaxScaler(feature_range=(0.1, 1.0)).fit(S_years[train_years].reshape(-1, num_sensors))
# Field: per-pixel min-max, as in the CS-SHRED repo.
sc_y = MinMaxScaler().fit(X_clean[train_years].reshape(-1, n_ocean))

S_scaled = sc_x.transform(S_years.reshape(-1, num_sensors)).reshape(S_years.shape).astype(np.float32)

# --- corrupt the sensors (missing = 0) ---------------------------------------------------
rng = np.random.default_rng(SEED)
missing = rng.random(S_scaled.shape) < SENSOR_TIME_DROP
missing |= np.isnan(S_scaled)
if SENSOR_DROP_FRAC > 0:
    dead = rng.choice(num_sensors, size=int(num_sensors * SENSOR_DROP_FRAC), replace=False)
    missing[:, :, dead] = True
S_masked = np.where(missing, 0.0, S_scaled).astype(np.float32)
print(f"Fraction of missing sensor samples: {missing.mean():.3f}")


def build_split(year_idx):
    '''Sliding lag windows inside each year -> X (n, lags, sensors), Y (n, n_ocean) scaled clean field.'''
    Xs, Ys = [], []
    for y in year_idx:
        w = sliding_window_view(S_masked[y], LAGS, axis=0)       # (days-lags+1, sensors, lags)
        Xs.append(np.ascontiguousarray(w.transpose(0, 2, 1)))     # -> (n, lags, sensors)
        Ys.append(sc_y.transform(X_clean[y, LAGS - 1:]).astype(np.float32))
    X = torch.from_numpy(np.concatenate(Xs, axis=0))
    Y = torch.from_numpy(np.concatenate(Ys, axis=0))
    return TimeSeriesDataset(X, Y)


train_dataset = build_split(train_years)
valid_dataset = build_split(valid_years)
test_dataset  = build_split(test_years)
n_state = n_ocean

for name, ds in [("train", train_dataset), ("valid", valid_dataset), ("test", test_dataset)]:
    print(f"{name:5s}: X {tuple(ds.X.shape)}  Y {tuple(ds.Y.shape)}")

years -> train 18, valid 3, test 5
Fraction of missing sensor samples: 0.491
train: X (6228, 20, 20)  Y (6228, 41027)
valid: X (1038, 20, 20)  Y (1038, 41027)
test : X (1730, 20, 20)  Y (1730, 41027)


## 3. Does the CS step actually fire?

`CSSHREDLAGS` runs SPGL1 on a (window, sensor) vector only if its fraction of zeros is
**greater than** `n_sparsity_threshold`. Check the fraction of vectors that qualify for a few thresholds.

In [18]:
zero_frac = (train_dataset.X == 0).float().mean(dim=1)          # (n_windows, n_sensors)
for thr in (0.75, 0.3, 0.2, 0.1):
    print(f"threshold {thr:4.2f}: CS triggered on {(zero_frac > thr).float().mean().item()*100:5.1f}% of (window, sensor) vectors")

threshold 0.75: CS triggered on  26.8% of (window, sensor) vectors
threshold 0.30: CS triggered on  57.1% of (window, sensor) vectors
threshold 0.20: CS triggered on  83.5% of (window, sensor) vectors
threshold 0.10: CS triggered on  97.6% of (window, sensor) vectors


## 4. Build & train CS-SHRED (`CSSHREDLAGS`)

Hyperparameters are the README defaults, except `n_sparsity_threshold`, which is set so that the CS
recovery is actually applied to a meaningful share of the windows (see the table above).

In [19]:
import time

hidden_size = 64
hidden_layers = 2
l1, l2 = 350, 400
dropout = 0.0

num_epochs = 200
batch_size = 32
lr = 1e-3
step_epoch = 5
patience = 15
verbose = True

# Adaptive-loss weights
lambL2, lambL1, lambdaSNR = 1.0, 0.1, 0.01

# Compressed-sensing (SPGL1) tolerances
l1_tol, opt_tol, ls_tol = 1e-4, 1e-5, 1e-5
N_SPARSITY_THRESHOLD = 0.2     # < SENSOR_TIME_DROP, otherwise CS never runs

model = models.CSSHREDLAGS(
    input_size=num_sensors,
    output_size=n_state,
    hidden_size=hidden_size,
    hidden_layers=hidden_layers,
    l1=l1, l2=l2,
    dropout=dropout,
    l1_tol=l1_tol, opt_tol=opt_tol, ls_tol=ls_tol,
    n_sparsity_threshold=N_SPARSITY_THRESHOLD,
    verbosity=-1,
    basis="fft",
    cache_max_entries=100000,
).to(device)



In [20]:
"""t0 = time.perf_counter()

train_error, validation_errors = models.fit_csshred_model(
    model, train_dataset, valid_dataset,
    batch_size=batch_size, num_epochs=num_epochs, lr=lr,
    lambL2=lambL2, lambL1=lambL1, lambdaSNR=lambdaSNR,
    step_epoch=step_epoch, verbose=verbose, patience=patience,
)

train_time = time.perf_counter() - t0
h, rem = divmod(int(train_time), 3600)
m, s = divmod(rem, 60)
n_epochs_run = len(train_error)
print(f"\nTraining time: {h:02d}h {m:02d}m {s:02d}s ({train_time:.1f} s) "
      f"| epochs run: {n_epochs_run} | {train_time / max(n_epochs_run, 1):.1f} s/epoch")"""

't0 = time.perf_counter()\n\ntrain_error, validation_errors = models.fit_csshred_model(\n    model, train_dataset, valid_dataset,\n    batch_size=batch_size, num_epochs=num_epochs, lr=lr,\n    lambL2=lambL2, lambL1=lambL1, lambdaSNR=lambdaSNR,\n    step_epoch=step_epoch, verbose=verbose, patience=patience,\n)\n\ntrain_time = time.perf_counter() - t0\nh, rem = divmod(int(train_time), 3600)\nm, s = divmod(rem, 60)\nn_epochs_run = len(train_error)\nprint(f"\nTraining time: {h:02d}h {m:02d}m {s:02d}s ({train_time:.1f} s) "\n      f"| epochs run: {n_epochs_run} | {train_time / max(n_epochs_run, 1):.1f} s/epoch")'

540 min

In [21]:
# save the model
if not Path("models").exists():
    Path("models").mkdir()
if not Path("models/mSHRED_model.pth").exists():
    torch.save(model.state_dict(), "models/mSHRED_model.pth")

## 5. Evaluate on the held-out test years

Metrics are computed in physical units (after `sc_y.inverse_transform`): MSE, relative RMSE
(`||Y - Ŷ|| / ||Y||`, no extra factor), mean per-snapshot relative error, plus PSNR/SSIM on the last snapshot.

In [26]:
def evaluate_model(model, test_dataset, sc_y, batch_size=64):
    if isinstance(model, models.CSSHREDLAGS):
        raw = models.forward_csshred_contiguous_batches(model, test_dataset.X, batch_size)
    else:
        model.eval()
        with torch.no_grad():
            raw = model(test_dataset.X)

    recons = sc_y.inverse_transform(raw.detach().cpu().numpy())
    truth = sc_y.inverse_transform(test_dataset.Y.detach().cpu().numpy())

    print(f"recons shape: {recons.shape}, truth shape: {truth.shape}")

    predicted_tensor = torch.as_tensor(recons, dtype=torch.float64)
    truth_tensor = torch.as_tensor(truth, dtype=torch.float64)
    if predicted_tensor.shape != truth_tensor.shape:
        raise ValueError(
            f"prediction shape {tuple(predicted_tensor.shape)} "
            f"does not match ground-truth shape {tuple(truth_tensor.shape)}"
        )

    error = predicted_tensor - truth_tensor
    mse_value = error.pow(2).mean(dim=-1).mean()
    mre_value = (
        error.pow(2).sum(dim=-1).sqrt()
        / truth_tensor.pow(2).sum(dim=-1).sqrt()
    ).mean()
    denominator = torch.amax(truth_tensor) - torch.amin(truth_tensor)

    metrics = {
        "nmae": torch.mean(torch.abs(error) / denominator).item()*0.206,
    }
    print({k: f"{v:.6f}" for k, v in metrics.items()})
    return metrics


print("Evaluating CS-SHRED model on all splits...")
train_m = evaluate_model(model, train_dataset, sc_y, batch_size=64)
valid_m = evaluate_model(model, valid_dataset, sc_y, batch_size=64)
test_m = evaluate_model(model, test_dataset, sc_y, batch_size=64)

import pandas as pd
df = pd.DataFrame({
    "model": ["(mSHRED): TR+cnn"],
    "type": ["(T.4.A): CSSHRED"],
    **{f"train-{k}": [v] for k, v in train_m.items()},
    **{f"valid-{k}": [v] for k, v in valid_m.items()},
    **{f"test-{k}": [v] for k, v in test_m.items()},
})
df.to_csv("dataframe_errors.csv", index=False)

Evaluating CS-SHRED model on all splits...
recons shape: (6228, 41027), truth shape: (6228, 41027)
{'nmae': '0.017618'}
recons shape: (1038, 41027), truth shape: (1038, 41027)
{'nmae': '0.019187'}
recons shape: (1730, 41027), truth shape: (1730, 41027)
{'nmae': '0.018010'}


## 6. Times
#### Time train 540 min
#### Time inference 63sec